# Create `YAML` Config File for `RBP ML` Input Data

## Purpose: 

Describe the configuration of: 
* `Cell Lines`
* `Peak Distance Thresholds` 
* `Data Creation Method` 

that went into creating each dataset and provide an easy-to-parse dictionary of all possible combinations of datasets.

## Packages and Options

In [1]:
import glob, os, yaml

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Literals

In [2]:
files = glob.glob("../final_modeling_input_datasets/*.tsv.gz")
yaml_dict = {}


## Define Covariates

In [3]:
cell_lines = sorted(list(set([file.split("/")[-1].split("_")[0] for file in files])))
cell_lines

yaml_dict["Cell Lines"] = cell_lines 

thresholds = sorted(list(set([file.split("/")[-1].split("_")[1] for file in files])))
thresholds

yaml_dict["Peak Distance Thresholds"] = thresholds

analysis_methods = sorted(list(set(["_".join(file.split("/")[-1].split(".")[0].split("_")[2:]) for file in files])))
analysis_methods

yaml_dict["Data Creation Methods"] = analysis_methods

['HepG2', 'K562']

['100', '1000', '10000', '2000', '50', '500', '5000']

['binary-binding-only',
 'expression-getmm_no-log_dose-dependent-expression',
 'expression-getmm_no-log_dose-independent-expression',
 'expression-getmm_yes-log_dose-dependent-expression',
 'expression-getmm_yes-log_dose-independent-expression',
 'expression-tmm_no-log_dose-dependent-expression',
 'expression-tmm_no-log_dose-independent-expression',
 'expression-tmm_yes-log_dose-dependent-expression',
 'expression-tmm_yes-log_dose-independent-expression',
 'num-peaks-only']

## Iterate Through All Combinations of Covariates and Label Data File

In [4]:
yaml_dict["Data Files"] = {}

encountered_files = set()

for cell_line in cell_lines: 
    yaml_dict["Data Files"][cell_line] = {}

    for threshold in thresholds: 
        yaml_dict["Data Files"][cell_line][threshold] = {}

        for analysis_method in analysis_methods:
            
            # get the file matching the covariates
            file = glob.glob("../final_modeling_input_datasets/{}_{}_{}*".format(cell_line, threshold, analysis_method))
            assert len(file)==1
            
            # make sure that we aren't picking up on duplicate files 
            assert os.path.abspath(file[0]) not in encountered_files
            # add the absolute path of the data file to files we've seen before  
            encountered_files.add(os.path.abspath(file[0]))
            
            # save absolute path of data file to dictionary 
            yaml_dict["Data Files"][cell_line][threshold][analysis_method] =  os.path.abspath(file[0])
            

## Create `YAML` File from `Dict`

In [5]:
with open("../final_modeling_input_datasets/data_config.yaml", 'w') as out_file: 
    _=yaml.dump(yaml_dict, out_file)

## Test `YAML` File `Dict` Conversion Ability

In [6]:
with open("../final_modeling_input_datasets/data_config.yaml", 'r') as yaml_file: 
    test_dict = yaml.safe_load(yaml_file)
    
test_dict

{'Cell Lines': ['HepG2', 'K562'],
 'Data Creation Methods': ['binary-binding-only',
  'expression-getmm_no-log_dose-dependent-expression',
  'expression-getmm_no-log_dose-independent-expression',
  'expression-getmm_yes-log_dose-dependent-expression',
  'expression-getmm_yes-log_dose-independent-expression',
  'expression-tmm_no-log_dose-dependent-expression',
  'expression-tmm_no-log_dose-independent-expression',
  'expression-tmm_yes-log_dose-dependent-expression',
  'expression-tmm_yes-log_dose-independent-expression',
  'num-peaks-only'],
 'Data Files': {'HepG2': {'100': {'binary-binding-only': '/sfs/gpfs/tardis/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/6_create_RBP_to_PSI_input_ML_datasets/final_modeling_input_datasets/HepG2_100_binary-binding-only.tsv.gz',
    'expression-getmm_no-log_dose-dependent-expression': '/sfs/gpfs/tardis/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/6_create_RBP_to_PSI_input_ML_d